# Generation, oversampling, and traceability

Generate synthetic rows and inspect their trace records.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from mimic import MIMIC, GenerationPolicy, RandomForestPathEncoder, MixedFeatureDecoder

rng = np.random.default_rng(2)
major = pd.DataFrame({
    "x": rng.normal(0, 1, 120),
    "y": rng.normal(0, 1, 120),
    "label": "majority",
})
minor = pd.DataFrame({
    "x": rng.normal(2.5, 0.5, 20),
    "y": rng.normal(2.5, 0.5, 20),
    "label": "minority",
})
df = pd.concat([major, minor], ignore_index=True)
df["id"] = np.arange(len(df))

display(df["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Original class balance"))
display(df.head(8).style.set_caption("Original training rows"))


In [ ]:
mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=6, random_state=2),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=2),
    policy=GenerationPolicy(method="smote", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 1.0)),
    n_bootstrap=2,
    random_state=2,
)
mimic.fit(df)
synthetic, trace = mimic.sample(12, return_trace=True)

display(synthetic.head(8).style.set_caption("Generated sample rows"))
display(trace.head(8).style.set_caption("Generation trace"))


In [ ]:
minority_needed = df["label"].value_counts()["majority"] - df["label"].value_counts()["minority"]
generated, generation_trace = mimic.sample(minority_needed * 10, return_trace=True)
minority_mask = generated["label"].eq("minority")
minority_synthetic = generated.loc[minority_mask].head(minority_needed).reset_index(drop=True)
minority_trace = generation_trace.loc[minority_mask.to_numpy()].head(minority_needed).reset_index(drop=True)
balanced = pd.concat([df.drop(columns=["id"]), minority_synthetic], ignore_index=True)

display(balanced["label"].value_counts().rename_axis("label").to_frame("count").style.set_caption("Class balance after minority oversampling"))
display(minority_synthetic.head(8).style.set_caption("Synthetic minority rows"))
display(minority_trace.head(8).style.set_caption("Oversampling trace for retained minority rows"))


In [ ]:
plot_df = pd.concat(
    [
        df.drop(columns=["id"]).assign(source="original"),
        minority_synthetic.assign(source="generated"),
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(6, 5))
for (source, label), part in plot_df.groupby(["source", "label"]):
    if source == "original" and label == "majority":
        color, marker, alpha, size = "#9aa0a6", "o", 0.35, 28
    elif source == "original" and label == "minority":
        color, marker, alpha, size = "#1f77b4", "o", 0.85, 42
    else:
        color, marker, alpha, size = "#ff7f0e", "x", 0.9, 52
    ax.scatter(part["x"], part["y"], label=f"{source} {label}", color=color, marker=marker, alpha=alpha, s=size)
ax.set_title("Minority oversampling in original 2D feature space")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(frameon=False)
fig.tight_layout()


In [ ]:
disp = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=6, random_state=3),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=3),
    policy=GenerationPolicy(method="displacement", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 1.0)),
    n_bootstrap=2,
    random_state=3,
)
disp.fit(df)
disp_synthetic, disp_trace = disp.sample(5, return_trace=True)

display(disp_synthetic.style.set_caption("Displacement-generated rows"))
display(disp_trace.style.set_caption("Displacement trace"))
